# Problem 5: Policy & Claims Copilot
## LLM + RAG (Retrieval-Augmented Generation) System

---
**Course:** L3-L5 Gen AI App Developer | Bronze Badge  
**Topic:** Generative AI, RAG Architecture, LLM Application Development  
**Type:** PROJECT + Presentation  

---

## Use Case: "Policy & Claims Copilot"
### Goal
Help **customers, agents, and claims teams** get **instant, consistent, grounded answers** about:
- What is covered / not covered in the policy
- Coverage limits and sub-limits
- Waiting periods
- Claim submission steps + timelines
- Documents required for claims
- **Pre-check a claim scenario** before official submission

### Why RAG (and NOT plain LLM)?
| Plain LLM | RAG-based LLM |
|-----------|---------------|
| May hallucinate/guess policy terms | Retrieves EXACT clauses from policy PDF |
| No source citation | **Quotes source section/page number** |
| Knowledge cut-off date | Uses **live policy documents** |
| Generic answers | **Grounded, policy-specific answers** |

### RAG Architecture Flow
```
Policy PDF
    ↓ Document Loader (PyPDF)
    ↓ Text Splitter (chunks)
    ↓ Embeddings (sentence-transformers)
    ↓ Vector Store (FAISS)
    ↓ Retriever (top-k similar chunks)
    ↓ LLM + Retrieved Context
    ↓ Grounded Answer + Source Citation
```

In [ ]:
# ============================================================
# STEP 1: Install Dependencies
# ============================================================
# Run this cell to install all required packages
# Then restart kernel and run from Step 2

import subprocess, sys

packages = [
    'langchain',            # Core RAG orchestration framework
    'langchain-community',  # Community integrations (FAISS, HuggingFace)
    'langchain-openai',     # OpenAI integration for LangChain
    'openai',               # OpenAI API client
    'faiss-cpu',            # Facebook AI Similarity Search (vector DB)
    'sentence-transformers',# Local embedding model (no API key needed)
    'pypdf',                # Read PDF files
    'python-dotenv',        # Load environment variables from .env file
    'tiktoken',             # Token counting for OpenAI models
    'transformers',         # HuggingFace transformers
    'gradio',               # Quick web UI for demos
]

print('Installing packages...')
for pkg in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    status = 'OK' if result.returncode == 0 else 'FAILED'
    print(f'  [{status}] {pkg}')

print('\nAll packages installed! Restart kernel if this is the first run.')

In [ ]:
# ============================================================
# STEP 2: Import Libraries & Configure Environment
# ============================================================

import os                          # OS interface
import warnings
warnings.filterwarnings('ignore')

# LangChain core components
from langchain.text_splitter import RecursiveCharacterTextSplitter  # Split text into overlapping chunks
from langchain.schema import Document                               # LangChain document wrapper

# Embeddings: Convert text to vectors
from langchain_community.embeddings import HuggingFaceEmbeddings    # Local embedding (no API key)

# Vector Store: FAISS — fast similarity search
from langchain_community.vectorstores import FAISS

# LLM providers (choose based on availability)
try:
    from langchain_openai import ChatOpenAI      # OpenAI GPT models
    OPENAI_AVAILABLE = True
except ImportError:
    OPENAI_AVAILABLE = False

# LangChain chains
from langchain.chains import RetrievalQA             # QA chain with retrieval
from langchain.prompts import PromptTemplate          # Custom prompts

# PDF loader
try:
    from langchain_community.document_loaders import PyPDFLoader  # Load real PDF
    PDF_LOADER_AVAILABLE = True
except ImportError:
    PDF_LOADER_AVAILABLE = False

import numpy as np
import matplotlib.pyplot as plt
import json
from typing import List, Dict, Optional

print('Step 2 COMPLETE: Libraries imported!')
print(f'  OpenAI available       : {OPENAI_AVAILABLE}')
print(f'  PDF Loader available   : {PDF_LOADER_AVAILABLE}')
print('NOTE: This notebook uses HuggingFace local embeddings (no API key needed for retrieval)')
print('For LLM responses: set OPENAI_API_KEY or use the mock LLM mode (built-in)')

In [ ]:
# ============================================================
# STEP 3: Create Sample Insurance Policy Document
# ============================================================
# In production: load the actual PDF file
# Here: create a realistic sample health insurance policy as text
# This simulates the content of a real policy document

SAMPLE_POLICY_TEXT = """
HEALTHSURE INSURANCE POLICY
Policy Type: Individual Health Insurance
Policy Number: HS-2024-TEMPLATE
Version: 2024.1
===================================================

SECTION 1: COVERAGE OVERVIEW
---------------------------------------------------
1.1 Sum Insured Options: Rs. 3 Lakhs, Rs. 5 Lakhs, Rs. 10 Lakhs, Rs. 25 Lakhs

1.2 What is Covered:
- Hospitalization expenses (minimum 24 hours continuous inpatient care)
- Pre-hospitalization expenses: 30 days prior to admission
- Post-hospitalization expenses: 60 days after discharge
- Day care procedures (186 listed procedures not requiring 24-hour admission)
- Emergency ambulance charges: up to Rs. 2,000 per hospitalization
- Domiciliary hospitalization: treatment at home for conditions requiring hospitalization
- Organ donor expenses: medical expenses of donor during organ transplant
- AYUSH treatments: Ayurveda, Yoga, Unani, Siddha, Homeopathy — up to Rs. 30,000
- Mental health treatment: hospitalization for mental illness covered

1.3 Sub-Limits:
- Room rent limit: 1% of Sum Insured per day (Single AC room)
- ICU charges: 2% of Sum Insured per day
- Cataract surgery: Rs. 40,000 per eye per policy year
- Knee replacement: Rs. 1,20,000 per knee
- Maternity benefit (if opted): Rs. 50,000 for normal delivery; Rs. 75,000 for C-section

SECTION 2: EXCLUSIONS (WHAT IS NOT COVERED)
---------------------------------------------------
2.1 Permanent Exclusions (never covered):
- Cosmetic or aesthetic treatment (unless necessitated by accident)
- Self-inflicted injuries and suicide attempts
- Treatment for alcoholism, drug abuse, substance misuse
- War, nuclear, biological or chemical hazards
- Dental treatment (unless requiring hospitalization due to accident)
- Spectacles, contact lenses, hearing aids
- Experimental or unproven treatments
- Treatment outside India (unless emergency)

2.2 Waiting Period Exclusions:
- Pre-existing diseases (PED): 36-month waiting period from policy start
- Specific listed ailments: 24-month waiting period
  (includes: hernia, piles/fistula, cataracts, benign ENT disorders, knee replacement,
   joint replacement, sinusitis, varicose veins)
- Initial waiting period: 30 days (except accidents)
- Maternity benefit: 24-month waiting period after opting

SECTION 3: CLAIMS PROCEDURE
---------------------------------------------------
3.1 Cashless Claim Process (Network Hospitals):
Step 1: Show HealthSure insurance card at hospital TPA desk
Step 2: Hospital sends pre-authorization request to TPA (at least 3 hours before planned procedure)
Step 3: TPA reviews and approves/denies within 2 hours (planned) or 1 hour (emergency)
Step 4: If approved, hospital bills are settled directly by insurance company
Step 5: Patient pays only non-covered expenses (co-pay, sub-limit excess, deductibles)

3.2 Reimbursement Claim Process (Non-Network Hospitals):
Step 1: Pay hospital bills entirely at discharge
Step 2: Collect all original bills, discharge summary, investigation reports
Step 3: Submit claim within 15 days of discharge
Step 4: TPA processes and responds within 7 working days
Step 5: Approved amount credited to registered bank account within 5 working days

3.3 Documents Required for All Claims:
- Duly filled Claim Form (available at TPA office or website)
- Original discharge summary from hospital
- All original bills (pharmacy, diagnostics, hospital)
- Indoor case papers / medical records
- Investigation reports (X-ray, MRI, blood reports, etc.)
- Doctor's prescription for all medicines
- Copy of photo ID (Aadhaar/Passport)
- Cancelled cheque or bank details for reimbursement

3.4 Additional Documents for Specific Cases:
- Accident cases: FIR/MLC (Medico Legal Certificate), police report
- Maternity: Birth certificate of newborn
- Death claim: Death certificate, post-mortem report (if applicable)
- Critical illness: Specialist's certificate confirming diagnosis

SECTION 4: TIMELINES & SLA
---------------------------------------------------
4.1 Intimation deadlines:
- Emergency hospitalization: within 24 hours of admission
- Planned hospitalization: 48 hours before admission
- Post-hospitalization claims: within 15 days of discharge

4.2 Claim Settlement timelines:
- Cashless approval (planned): within 2 hours of receiving pre-authorization
- Cashless approval (emergency): within 1 hour
- Reimbursement: 7 working days for initial review; 30 days for complete settlement
- Claim deficiency notice: within 7 working days of receiving incomplete documents

4.3 Claim rejection timelines:
- Insurer must communicate rejection with reason within 30 days of receiving all documents
- Policyholder can appeal within 15 days of rejection notice

SECTION 5: PREMIUM & RENEWAL
---------------------------------------------------
5.1 Premium payment modes: Annual, Semi-Annual, Quarterly
5.2 Grace period for renewal: 30 days after premium due date
5.3 No-claim bonus: 5% increase in Sum Insured for each claim-free year (max 50%)
5.4 Mid-term cancellation: Pro-rata refund subject to no claims in current year
5.5 Free look period: 15 days from policy receipt for review and cancellation

SECTION 6: CO-PAYMENT & DEDUCTIBLES
---------------------------------------------------
6.1 Co-payment options:
- 0% co-pay: standard premium
- 10% co-pay: 5% premium discount
- 20% co-pay: 10% premium discount
6.2 Compulsory co-pay: 20% for insured persons aged 60+ years at entry
6.3 Zone-based pricing: Zone A (metro cities) > Zone B (Tier 2) > Zone C (other)

SECTION 7: NETWORK HOSPITALS
---------------------------------------------------
7.1 Network hospitals: Over 8,000 hospitals across India
7.2 Cashless facility available only at listed network hospitals
7.3 Updated network list available at: www.healthsure.in/network
7.4 If admitted to non-network hospital in emergency: reimbursement applicable

SECTION 8: CONTACT INFORMATION
---------------------------------------------------
8.1 TPA Helpline (24x7): 1800-XXX-YYYY (toll-free)
8.2 Email for claims: claims@healthsure.in
8.3 Grievance: grievance@healthsure.in
8.4 IRDAI Helpline: 155255
"""

# Save policy to a text file (simulating a document)
with open('health_insurance_policy.txt', 'w', encoding='utf-8') as f:
    f.write(SAMPLE_POLICY_TEXT)

print(f'Policy document created: health_insurance_policy.txt')
print(f'Document length: {len(SAMPLE_POLICY_TEXT)} characters')
print(f'Sections covered: 8 (Coverage, Exclusions, Claims, Timelines, Premium, Co-pay, Network, Contact)')

In [ ]:
# ============================================================
# STEP 4: Load Document and Split into Chunks
# ============================================================
# WHY CHUNKING?
# - LLMs have limited context windows (e.g., 4096 tokens)
# - We can't pass the entire policy document to the LLM
# - Solution: split into overlapping chunks, retrieve only relevant ones
# - Overlap ensures boundary information is not lost between chunks

# Load the policy text as LangChain Documents
with open('health_insurance_policy.txt', 'r', encoding='utf-8') as f:
    raw_text = f.read()

# Wrap in LangChain Document format
full_doc = Document(
    page_content=raw_text,
    metadata={'source': 'health_insurance_policy.txt', 'type': 'insurance_policy'}
)

# Text Splitter Configuration
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,          # Each chunk = ~500 characters (approx 100-120 tokens)
    chunk_overlap=80,        # 80 characters overlap between consecutive chunks
    length_function=len,     # Use character count for chunk size measurement
    separators=[             # Split priority: sections > paragraphs > sentences > words
        '\n\n',              # First split at double newlines (paragraphs)
        '\n',                # Then single newlines (lines)
        '.',                 # Then sentences
        ' ',                 # Then words (last resort)
    ]
)

# Split the document into chunks
chunks = splitter.split_documents([full_doc])

print(f'Document loaded and split into {len(chunks)} chunks')
print(f'Chunk size: ~{splitter._chunk_size} chars | Overlap: {splitter._chunk_overlap} chars')
print('\nSample chunks:')
for i, chunk in enumerate(chunks[:3]):  # Show first 3 chunks
    print(f'\n--- Chunk {i+1} ({len(chunk.page_content)} chars) ---')
    print(chunk.page_content[:200] + '...' if len(chunk.page_content) > 200 else chunk.page_content)
    print(f'Metadata: {chunk.metadata}')

In [ ]:
# ============================================================
# STEP 5: Create Embeddings and Build FAISS Vector Store
# ============================================================
# EMBEDDINGS: Convert text chunks to numeric vectors (dense embeddings)
# - Each chunk → a vector of e.g. 384 or 768 dimensions
# - Semantically similar text → vectors close in embedding space
# - Model: all-MiniLM-L6-v2 (fast, 384-dim, good for English Q&A)
#
# FAISS: Facebook AI Similarity Search
# - Indexes all chunk vectors
# - At query time: converts query to vector, finds top-k nearest chunks
# - Uses cosine similarity or L2 distance

print('Loading embedding model: all-MiniLM-L6-v2 (sentence-transformers)...')
print('(First run downloads ~90MB model — subsequent runs use cached version)')

# Initialize HuggingFace local embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',  # 384-dimensional embeddings
    model_kwargs={'device': 'cpu'},   # Use CPU (change to 'cuda' if GPU available)
    encode_kwargs={'normalize_embeddings': True}  # L2 normalize → enables cosine similarity
)

print('Embedding model loaded!')

# Test embedding with a sample sentence
test_vec = embedding_model.embed_query('What is covered under hospitalization?')
print(f'Test embedding vector: dimension = {len(test_vec)}')

# Build FAISS vector store from all chunks
print(f'\nBuilding FAISS index for {len(chunks)} chunks...')
vectorstore = FAISS.from_documents(
    documents=chunks,           # Policy text chunks
    embedding=embedding_model   # Embedding model to convert text to vectors
)

# Save vector store to disk for future use (no need to rebuild every time)
vectorstore.save_local('faiss_policy_index')  # Saves index files locally

print(f'FAISS vector store built and saved to: faiss_policy_index/')
print(f'Total vectors indexed: {len(chunks)}')

# Test retrieval
test_docs = vectorstore.similarity_search('What documents are needed for a claim?', k=2)
print(f'\nTest retrieval: Top 2 chunks for "What documents are needed for a claim?"')
for i, doc in enumerate(test_docs):
    print(f'  Chunk {i+1}: {doc.page_content[:150]}...')

In [ ]:
# ============================================================
# STEP 6: Build the RAG Retriever + Prompt Template
# ============================================================
# The retriever: given a query, finds the most relevant policy chunks
# The prompt template: formats the retrieved chunks + query for the LLM

# Create a retriever from the vector store
retriever = vectorstore.as_retriever(
    search_type='similarity',   # Find most similar chunks (cosine similarity)
    search_kwargs={'k': 4}      # Retrieve top 4 most relevant chunks per query
)

# Custom RAG Prompt Template
# The {context} placeholder will be filled with retrieved policy chunks
# The {question} placeholder will be filled with the user's question
RAG_PROMPT_TEMPLATE = """
You are a knowledgeable Insurance Policy Assistant for HealthSure Insurance.
Your role is to answer questions ONLY based on the provided policy document context.

RULES:
1. Answer ONLY from the context below. Do NOT use outside knowledge.
2. If the answer is not in the context, say: "This information is not found in the provided policy document."
3. Always cite the relevant section number when answering (e.g., "Per Section 3.2...")
4. Be concise but complete. Use bullet points for lists.
5. For claim pre-checks: state clearly if the scenario is likely COVERED or NOT COVERED with reasoning.

POLICY CONTEXT (retrieved relevant sections):
---------------------------------------------
{context}
---------------------------------------------

CUSTOMER QUESTION: {question}

ANSWER (cite section numbers):
"""

# Create PromptTemplate object
rag_prompt = PromptTemplate(
    template=RAG_PROMPT_TEMPLATE,
    input_variables=['context', 'question']  # Variables filled at runtime
)

print('RAG Retriever configured!')
print(f'Retrieval: top {retriever.search_kwargs["k"]} chunks per query')
print('Prompt template: grounded, section-citing, claim pre-check capable')

# Show a retrieval example
sample_query = 'What is the waiting period for pre-existing diseases?'
retrieved = retriever.get_relevant_documents(sample_query)
print(f'\nSample retrieval for: "{sample_query}"')
for i, doc in enumerate(retrieved[:2]):
    print(f'  [{i+1}] {doc.page_content[:200]}...')

In [ ]:
# ============================================================
# STEP 7: Configure LLM (OpenAI or Mock Mode)
# ============================================================
# Option A: Use OpenAI GPT (requires OPENAI_API_KEY)
# Option B: Use Mock LLM (built-in rule-based fallback for demo)
#
# To use OpenAI:
# 1. Create a .env file with: OPENAI_API_KEY=sk-xxxxxxxxxxxx
# 2. Or set: os.environ['OPENAI_API_KEY'] = 'your-key-here'

# Try to load API key from environment or .env file
try:
    from dotenv import load_dotenv
    load_dotenv()   # Loads .env file in current directory
except ImportError:
    pass

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')  # Gets key from environment

if OPENAI_API_KEY and OPENAI_AVAILABLE:
    # Use real OpenAI GPT-3.5-Turbo
    llm = ChatOpenAI(
        model='gpt-3.5-turbo',    # Model name
        temperature=0.1,           # Low temperature = more deterministic, factual answers
        max_tokens=500,            # Limit output length
        openai_api_key=OPENAI_API_KEY
    )
    LLM_MODE = 'OpenAI GPT-3.5-Turbo'
    print(f'LLM Mode: {LLM_MODE}')
else:
    # Mock LLM: Retrieval-only demo (shows retrieved chunks as answer)
    # This works WITHOUT any API key for demonstration purposes
    llm = None
    LLM_MODE = 'Mock (Retrieval-Only Demo)'
    print(f'LLM Mode: {LLM_MODE}')
    print('To enable real LLM responses: set OPENAI_API_KEY environment variable')
    print('Alternatively: install ollama (free, local) and use Ollama integration')

print(f'\nLLM configured in mode: {LLM_MODE}')

In [ ]:
# ============================================================
# STEP 8: Build the Complete RAG Q&A Function
# ============================================================
# The full pipeline:
# User Query → Embeddings → FAISS Retrieval → Top-k Chunks
# → Prompt (Context + Query) → LLM → Grounded Answer

def rag_answer(question: str, top_k: int = 4) -> Dict:
    """
    Answer a policy question using RAG.

    Parameters:
        question : str  — Customer's natural language question
        top_k    : int  — Number of chunks to retrieve (default 4)

    Returns:
        dict with keys: question, answer, sources, chunks
    """

    # Step 8a: Retrieve relevant chunks
    docs = vectorstore.similarity_search_with_score(question, k=top_k)
    # Returns list of (Document, similarity_score) tuples

    # Step 8b: Format context from retrieved chunks
    context_parts = []
    sources = []
    for doc, score in docs:
        context_parts.append(doc.page_content)
        sources.append({
            'content': doc.page_content[:200] + '...',  # Truncate for display
            'source':  doc.metadata.get('source', 'policy_document'),
            'similarity_score': round(float(score), 4)
        })

    context_text = '\n\n'.join(context_parts)  # Join chunks with blank lines

    # Step 8c: Generate answer using LLM (or mock)
    if llm is not None:
        # Format the full prompt with context + question
        formatted_prompt = rag_prompt.format(context=context_text, question=question)

        # Call the LLM
        response = llm.invoke(formatted_prompt)
        answer = response.content if hasattr(response, 'content') else str(response)
    else:
        # Mock mode: format retrieved chunks as a structured answer
        answer = (
            f'[RETRIEVAL-ONLY MODE - No LLM API Key configured]\n\n'
            f'Based on the retrieved policy sections for your question:\n'
            f'"{question}"\n\n'
            f'The following relevant policy text was found:\n\n'
        )
        for i, (doc, score) in enumerate(docs[:2]):
            answer += f'[Source {i+1} | Relevance: {1 - score:.2%}]\n'
            answer += doc.page_content + '\n\n'

    return {
        'question': question,
        'answer':   answer,
        'sources':  sources,
        'chunks_retrieved': len(docs)
    }

print('RAG Q&A function built!')
print('Pipeline: Query → Embeddings → FAISS retrieval → Prompt + Context → LLM → Answer')

In [ ]:
# ============================================================
# STEP 9: Demo — Run Sample Q&A Queries
# ============================================================
# Test the RAG system with typical customer questions

sample_questions = [
    'What is the waiting period for pre-existing diseases?',
    'What documents do I need to submit a reimbursement claim?',
    'Is dental treatment covered under this policy?',
    'How much is covered for cataract surgery?',
    'What is the deadline to submit a claim after discharge?',
    'Can I get cashless treatment at any hospital?',
    'Is mental health treatment covered?',
    'What happens if I miss the policy renewal date?'
]

print('=== POLICY Q&A DEMO ===')
print('='*70)

all_results = []   # Store all Q&A results for later display

for q in sample_questions:
    print(f'\nQ: {q}')
    result = rag_answer(q, top_k=3)   # Get answer using RAG
    all_results.append(result)

    # Display answer (truncated for readability)
    answer_preview = result['answer'][:400] + '...' if len(result['answer']) > 400 else result['answer']
    print(f'A: {answer_preview}')

    # Show source information
    if result['sources']:
        best_source = result['sources'][0]
        print(f'   [Top Source | Similarity Score: {best_source["similarity_score"]}]')
    print('-'*70)

print(f'\nCompleted {len(sample_questions)} Q&A queries!')

In [ ]:
# ============================================================
# STEP 10: Claim Pre-Check Feature
# ============================================================
# Pre-check: analyze a claim scenario BEFORE official submission
# This helps patients understand if their claim is likely to be approved
# Reduces rejected claims and improves first-time-right submissions

def claim_precheck(scenario: str) -> Dict:
    """
    Pre-check a claim scenario against the policy.

    Parameters:
        scenario : str — Description of the medical situation and claim

    Returns:
        dict with verdict, reasoning, and documents needed
    """

    # Enhanced prompt specifically for claim pre-checks
    precheck_question = (
        f'CLAIM PRE-CHECK REQUEST:\n'
        f'Scenario: {scenario}\n\n'
        f'Please evaluate:\n'
        f'1. Is this scenario COVERED under the policy? (YES/NO/PARTIALLY)\n'
        f'2. What specific clause supports your verdict?\n'
        f'3. Are there any waiting period concerns?\n'
        f'4. What documents will be needed for this claim?\n'
        f'5. Any sub-limits or deductibles that apply?'
    )

    # Retrieve relevant policy chunks
    docs = vectorstore.similarity_search_with_score(precheck_question, k=5)

    context = '\n\n'.join([d.page_content for d, s in docs])

    if llm is not None:
        formatted = rag_prompt.format(context=context, question=precheck_question)
        response = llm.invoke(formatted)
        answer = response.content if hasattr(response, 'content') else str(response)
    else:
        # Mock pre-check: analyze based on keywords in scenario and retrieved context
        answer = generate_mock_precheck(scenario, context, docs)

    return {
        'scenario': scenario,
        'precheck_result': answer,
        'retrieved_sections': len(docs)
    }


def generate_mock_precheck(scenario: str, context: str, docs) -> str:
    """
    Rule-based mock pre-check when LLM is not available.
    Checks for common keywords to determine coverage status.
    """
    scenario_lower = scenario.lower()
    context_lower  = context.lower()

    # Simple keyword-based rules
    exclusion_keywords = ['cosmetic', 'dental', 'spectacles', 'alcohol', 'self-inflicted',
                          'drug abuse', 'war', 'experimental']
    covered_keywords   = ['hospitalization', 'surgery', 'emergency', 'accident', 'icu',
                          'day care', 'organ', 'maternity']
    ped_keywords       = ['pre-existing', 'diabetes', 'hypertension', 'heart disease', 'cancer']

    if any(kw in scenario_lower for kw in exclusion_keywords):
        verdict = 'LIKELY NOT COVERED'
        reason  = 'The scenario involves a condition listed as a permanent exclusion.'
    elif any(kw in scenario_lower for kw in ped_keywords):
        verdict = 'SUBJECT TO WAITING PERIOD'
        reason  = 'Pre-existing conditions have a 36-month waiting period per Section 2.2.'
    elif any(kw in scenario_lower for kw in covered_keywords):
        verdict = 'LIKELY COVERED'
        reason  = 'The scenario matches covered conditions per Section 1.2.'
    else:
        verdict = 'REQUIRES REVIEW'
        reason  = 'Insufficient information to determine coverage. Please contact TPA.'

    # Find relevant context excerpt
    context_excerpt = docs[0][0].page_content[:300] if docs else 'No relevant section found.'

    result = (
        f'[CLAIM PRE-CHECK RESULT - Mock Analysis]\n'
        f'{'='*50}\n'
        f'VERDICT: {verdict}\n\n'
        f'Reasoning: {reason}\n\n'
        f'Relevant Policy Section:\n{context_excerpt}\n\n'
        f'DISCLAIMER: This is a pre-check only. Final determination by TPA.\n'
        f'Contact: 1800-XXX-YYYY for official claim assessment.'
    )
    return result


# Run sample claim pre-checks
claim_scenarios = [
    'I was diagnosed with diabetes 2 years ago and now need kidney dialysis. Policy is 1 year old.',
    'I had an emergency appendectomy last week. I have been insured for 3 years.',
    'My mother needs cataract surgery. She has been covered for 18 months.',
    'I need dental implants after a car accident injury.',
    'My child needs treatment for a mental health condition and requires 5 days hospitalization.'
]

print('=== CLAIM PRE-CHECK DEMO ===')
print('='*70)

for i, scenario in enumerate(claim_scenarios, 1):
    print(f'\nScenario {i}: {scenario}')
    result = claim_precheck(scenario)
    print(f'Pre-check Result:')
    print(result['precheck_result'][:500] + '...' if len(result['precheck_result']) > 500
          else result['precheck_result'])
    print('-'*70)

In [ ]:
# ============================================================
# STEP 11: Visualize the RAG Pipeline Architecture
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# ---- Plot 1: RAG Pipeline Flow Diagram ----
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 12)
ax.axis('off')

# Pipeline stages: (label, x_center, y_center, width, height, color)
stages = [
    ('POLICY PDF / TXT\n(Source Document)',        5, 11.0, 5.5, 0.7, '#3498db'),
    ('DOCUMENT LOADER\n(PyPDF / TextLoader)',       5,  9.5, 5.5, 0.7, '#2980b9'),
    ('TEXT SPLITTER\n(chunk_size=500, overlap=80)', 5,  8.0, 5.5, 0.7, '#1abc9c'),
    ('EMBEDDING MODEL\n(all-MiniLM-L6-v2)',         5,  6.5, 5.5, 0.7, '#27ae60'),
    ('FAISS VECTOR STORE\n(Indexed Chunk Vectors)', 5,  5.0, 5.5, 0.7, '#f39c12'),
    ('CUSTOMER QUERY',                              5,  3.5, 5.5, 0.7, '#e74c3c'),
    ('RETRIEVER\n(Top-4 Similar Chunks)',            5,  2.0, 5.5, 0.7, '#9b59b6'),
    ('LLM + RAG PROMPT\n(Context + Question)',       5,  0.7, 5.5, 0.7, '#8e44ad'),
]

for label, cx, cy, w, h, color in stages:
    rect = plt.Rectangle((cx - w/2, cy - h/2), w, h,
                          facecolor=color, edgecolor='black', linewidth=1.5,
                          alpha=0.85, zorder=3)
    ax.add_patch(rect)
    ax.text(cx, cy, label, ha='center', va='center',
            fontsize=8.5, fontweight='bold', color='white', zorder=5)

# Draw arrows between stages
for i in range(len(stages) - 1):
    _, _, y_curr, _, h_curr, _ = stages[i]
    _, _, y_next, _, h_next, _ = stages[i+1]
    # Skip arrow from FAISS to CUSTOMER (break in pipeline)
    if i == 4: continue
    ax.annotate('', xy=(5, y_next + h_next/2), xytext=(5, y_curr - h_curr/2),
                arrowprops=dict(arrowstyle='->', color='black', lw=2))

# Label the output
ax.text(5, -0.1, 'GROUNDED ANSWER + SOURCE CITATIONS',
        ha='center', va='top', fontsize=9, fontweight='bold', color='#c0392b')

ax.set_title('RAG Pipeline Architecture\nPolicy & Claims Copilot',
             fontsize=12, fontweight='bold')

# ---- Plot 2: Chunk Retrieval Similarity Scores ----
ax2 = axes[1]

# Demo: show similarity scores for different queries
demo_queries = [
    'waiting period pre-existing disease',
    'documents required claim submission',
    'dental treatment exclusion',
    'cashless hospital network'
]

all_scores = []
for q in demo_queries:
    results = vectorstore.similarity_search_with_score(q, k=3)
    # FAISS returns L2 distances (lower = more similar)
    # Convert to similarity: sim = 1 / (1 + L2_distance)
    scores = [1 / (1 + s) for _, s in results]
    all_scores.append(scores)

x_pos = np.arange(len(demo_queries))
width = 0.25
colors_bar = ['#3498db', '#27ae60', '#e74c3c']

for i in range(3):
    chunk_scores = [scores[i] if i < len(scores) else 0 for scores in all_scores]
    ax2.bar(x_pos + i * width, chunk_scores, width,
            label=f'Chunk {i+1}', color=colors_bar[i], edgecolor='black', alpha=0.8)

ax2.set_xlabel('Query Topic', fontsize=10)
ax2.set_ylabel('Retrieval Similarity Score', fontsize=10)
ax2.set_title('FAISS Retrieval Similarity Scores\n(Higher = More Relevant Chunk)',
              fontsize=12, fontweight='bold')
ax2.set_xticks(x_pos + width)
short_queries = ['Waiting\nPeriod', 'Claim\nDocs', 'Dental\nExclusion', 'Cashless\nHospital']
ax2.set_xticklabels(short_queries, fontsize=9)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, axis='y')
ax2.set_ylim(0, 1.0)

plt.suptitle('Policy & Claims Copilot: RAG System Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('rag_architecture_visualization.png', dpi=120, bbox_inches='tight')
plt.show()
print('RAG architecture diagram saved: rag_architecture_visualization.png')

In [ ]:
# ============================================================
# STEP 12: Optional — Launch Gradio Web Interface
# ============================================================
# Gradio creates a web UI accessible in your browser
# Allows interactive Q&A with the RAG system without coding
# Run this cell to launch the interface (Ctrl+C in terminal to stop)

try:
    import gradio as gr   # Gradio: rapid web UI builder

    def policy_copilot_ui(question: str, mode: str) -> str:
        """
        Gradio interface function.
        question : customer's question
        mode     : 'Q&A' or 'Claim Pre-Check'
        """
        if not question.strip():
            return 'Please enter a question.'

        if mode == 'Claim Pre-Check':
            result = claim_precheck(question)    # Pre-check mode
            return result['precheck_result']
        else:
            result = rag_answer(question)         # Standard Q&A mode
            answer = result['answer']
            # Append source info
            answer += '\n\n--- SOURCES RETRIEVED ---'
            for i, src in enumerate(result['sources'][:3]):
                answer += f'\nSource {i+1}: {src["content"][:150]}...'
            return answer

    # Build Gradio interface
    demo = gr.Interface(
        fn=policy_copilot_ui,   # Function to call
        inputs=[
            gr.Textbox(
                lines=3,
                placeholder='Ask about your insurance policy... (e.g., What documents do I need to file a claim?)',
                label='Your Question'
            ),
            gr.Radio(
                ['Policy Q&A', 'Claim Pre-Check'],
                value='Policy Q&A',
                label='Mode'
            )
        ],
        outputs=gr.Textbox(lines=15, label='Copilot Answer'),
        title='HealthSure Policy & Claims Copilot',
        description=(
            'Get instant answers about your insurance policy. '
            'Ask about coverage, exclusions, claim procedures, or do a pre-check of your claim scenario.'
        ),
        examples=[
            ['What is the waiting period for pre-existing diseases?', 'Policy Q&A'],
            ['Is mental health treatment covered?', 'Policy Q&A'],
            ['I need knee replacement surgery. Policy is 3 years old.', 'Claim Pre-Check'],
            ['I was in a car accident and need dental treatment. Am I covered?', 'Claim Pre-Check'],
        ],
        theme=gr.themes.Soft()
    )

    # Launch the web interface
    # share=False: local only; share=True: creates public URL (for sharing)
    print('Launching Gradio interface...')
    print('Open the URL shown below in your browser to access the UI')
    demo.launch(share=False, inbrowser=True)   # Opens browser automatically

except ImportError:
    print('Gradio not installed. Install with: pip install gradio')
    print('The RAG Q&A functions are still available — use rag_answer() and claim_precheck() directly')

# STEP 13: System Architecture, Design Decisions & Presentation Guide

---

## Complete Architecture

```
┌─────────────────────────────────────────────────────────┐
│                  OFFLINE (Indexing Phase)               │
│  Policy PDF → PyPDF Loader → Text Splitter (500 chars)  │
│       → all-MiniLM-L6-v2 Embeddings → FAISS Index      │
└─────────────────────────────────────────────────────────┘
                        ↓ index saved
┌─────────────────────────────────────────────────────────┐
│                  ONLINE (Query Phase)                   │
│  User Question → Embed Query → FAISS Top-4 Search       │
│  → Retrieved Chunks + Question → RAG Prompt             │
│  → GPT-3.5 / LLM → Grounded Answer + Section Cites     │
│                   → Display to User                     │
└─────────────────────────────────────────────────────────┘
```

---

## Key Design Decisions

### 1. Why sentence-transformers/all-MiniLM-L6-v2?
- **Free** (no API key needed), runs locally
- 384 dimensions — compact but semantically rich
- Trained on MS-MARCO (Q&A retrieval data)
- 5x faster than larger models with minimal quality loss

### 2. Why FAISS over ChromaDB?
- FAISS: optimized for pure vector search, battle-tested at scale
- ChromaDB: easier metadata filtering, better for hybrid search
- For pure semantic retrieval on insurance policies: **FAISS is preferred**

### 3. Why chunk_size=500 with overlap=80?
- Too small (< 200): chunks lack context, answers incomplete
- Too large (> 1000): retriever returns too much noise
- 500 chars ≈ 1–2 policy clauses per chunk (ideal granularity)
- 80-char overlap: ensures clause boundary information is not lost

### 4. Why temperature=0.1 for LLM?
- **Low temperature = deterministic, factual** responses
- Insurance is a regulated domain; creative/random answers are risky
- We want the LLM to stick closely to retrieved policy text

---

## Business Impact

| Metric | Before Copilot | After Copilot (projected) |
|--------|---------------|---------------------------|
| Call center volume | 100% | -40% (handled by copilot) |
| Claim first-submission acceptance | ~65% | ~85% (pre-check feature) |
| Response time | 2–5 min wait | Instant |
| Answer consistency | Variable (agent-dependent) | Consistent (grounded in policy) |
| 24x7 availability | No | Yes |

---

## Presentation Talking Points

1. **Problem**: Customers don't read 40-page policy documents → call centers overwhelmed
2. **Solution**: RAG-based copilot answers from the exact policy text
3. **Why RAG over plain GPT**: No hallucination, source-cited, always current
4. **Technical stack**: LangChain + FAISS + sentence-transformers + GPT-3.5
5. **Demo**: Live Q&A + claim pre-check on the Gradio interface
6. **Scalability**: Add more PDFs (endorsements, circulars) → re-index FAISS
7. **Future**: Multi-language support, WhatsApp integration, voice interface